# Batch Segmentation with SAM2 — Kaggle

Combined notebook: `SegmentationModule` class + batch processing pipeline.

In [ ]:
# Debug: verify input datasets are mounted
!ls /kaggle/input/
!echo "---"
!ls /kaggle/input/datasets/ifty3110/segmentation-module-checkpoints-config/checkpoints/
!echo "---"
!ls /kaggle/input/datasets/shaidurpranto/capstone-demonstration-inputs

In [ ]:
!pip install git+https://github.com/facebookresearch/sam2.git --quiet

In [ ]:
!pip install omegaconf==2.3.0 hydra-core==1.3.2 --quiet

In [ ]:
import sys
sys.path = [p for p in sys.path if 'segmentation-module' not in p]

## Configuration

In [ ]:
import os

# =============================
# CONFIGURATION
# =============================
INPUT_DIR  = "/kaggle/input/datasets/shaidurpranto/capstone-demonstration-inputs"
OUTPUT_DIR = "/kaggle/working/capstone-demonstration-outputs"

CHECKPOINT  = "/kaggle/input/datasets/ifty3110/segmentation-module-checkpoints-config/checkpoints/sam2_hiera_large.pt"
MODEL_CONFIG = "sam2_hiera_l.yaml"  # SAM2 resolves this from its installed package

SUPPORTED_EXTENSIONS = (".jpg", ".jpeg", ".png")

# Output sub-directories
COORDS_DIR   = os.path.join(OUTPUT_DIR, "coordinates")
MASKS_DIR    = os.path.join(OUTPUT_DIR, "masks")
SEGMENTS_DIR = os.path.join(OUTPUT_DIR, "segments")

for d in [COORDS_DIR, MASKS_DIR, SEGMENTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Input   : {INPUT_DIR}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Checkpoint exists: {os.path.exists(CHECKPOINT)}")

## Imports

In [ ]:
import json
import time
import cv2
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## SegmentationModule Class

In [ ]:
class SegmentationModule:
    """
    A segmentation module for food image segmentation using SAM2.
    """

    def __init__(self, checkpoint=CHECKPOINT, model_config=MODEL_CONFIG, device=DEVICE):
        """
        Initialize the SegmentationModule.

        Args:
            checkpoint   (str): Path to the SAM2 .pt checkpoint file.
            model_config (str): SAM2 config filename (e.g. 'sam2_hiera_l.yaml').
            device       (str): 'cuda' or 'cpu'.
        """
        self.checkpoint   = checkpoint
        self.model_config = model_config
        self.device       = device

        # SAM2 mask-generator parameters
        self.points_per_side        = 32
        self.points_per_batch       = 32
        self.pred_iou_thresh        = 0.8
        self.stability_score_thresh = 0.92
        self.box_nms_thresh         = 0.6
        self.min_mask_region_area   = 2000
        self.dedup_iou_threshold    = 0.85
        self.min_area_ratio         = 0.20

        self.mask_generator = self._load_model()

    # ------------------------------------------------------------------
    # PRIVATE: Model loading
    # ------------------------------------------------------------------

    def _load_model(self):
        if not os.path.exists(self.checkpoint):
            raise FileNotFoundError(f"SAM2 checkpoint not found at {self.checkpoint}")

        print(f"Loading SAM2 on {self.device}...")
        sam2 = build_sam2(
            self.model_config, self.checkpoint,
            device=self.device, apply_postprocessing=True
        )
        return SAM2AutomaticMaskGenerator(
            model=sam2,
            points_per_side=self.points_per_side,
            points_per_batch=self.points_per_batch,
            pred_iou_thresh=self.pred_iou_thresh,
            stability_score_thresh=self.stability_score_thresh,
            box_nms_thresh=self.box_nms_thresh,
            min_mask_region_area=self.min_mask_region_area,
        )

    # ------------------------------------------------------------------
    # PRIVATE: Helpers
    # ------------------------------------------------------------------

    def _load_image_rgb(self, input):
        """
        Accept a file path (str) or a PIL Image and return an RGB numpy array.
        """
        if isinstance(input, str):
            if not os.path.exists(input):
                raise FileNotFoundError(f"Image not found at {input}")
            image = cv2.imdecode(np.fromfile(input, dtype=np.uint8), cv2.IMREAD_COLOR)
            if image is None:
                raise ValueError(f"Failed to read image: {input}")
            return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        elif isinstance(input, Image.Image):
            return np.array(input.convert("RGB"))
        else:
            raise TypeError("Input must be a file path (str) or a PIL.Image object")

    def _generate_masks(self, image_rgb):
        """Run SAM2 and apply area-based filtering + deduplication."""
        masks = self.mask_generator.generate(image_rgb)
        masks = self._deduplicate_masks(masks)
        if masks:
            max_area = max(m["area"] for m in masks)
            masks = [m for m in masks if m["area"] >= self.min_area_ratio * max_area]
        return masks

    def _deduplicate_masks(self, masks):
        """Remove heavily overlapping masks, keeping the larger one."""
        masks = sorted(masks, key=lambda m: m["area"], reverse=True)
        kept = []
        for candidate in masks:
            c_mask = candidate["segmentation"]
            is_duplicate = False
            for kept_mask in kept:
                k_mask = kept_mask["segmentation"]
                intersection = np.logical_and(c_mask, k_mask).sum()
                union        = np.logical_or(c_mask,  k_mask).sum()
                if union > 0 and (intersection / union) > self.dedup_iou_threshold:
                    is_duplicate = True
                    break
            if not is_duplicate:
                kept.append(candidate)
        return kept

    # ------------------------------------------------------------------
    # PUBLIC: Input methods
    # ------------------------------------------------------------------

    def set_image_from_path(self, image_path):
        """Load and cache an image from a file path."""
        self._cached_image_rgb = self._load_image_rgb(image_path)
        self._cached_masks = None

    def set_image(self, image):
        """Load and cache a PIL Image."""
        self._cached_image_rgb = self._load_image_rgb(image)
        self._cached_masks = None

    def _get_cached_masks(self, input=None):
        """Return cached masks, generating them if not yet computed."""
        if input is not None:
            image_rgb = self._load_image_rgb(input)
            return image_rgb, self._generate_masks(image_rgb)

        if not hasattr(self, "_cached_image_rgb") or self._cached_image_rgb is None:
            raise RuntimeError(
                "No image loaded. Call set_image() or set_image_from_path() first, "
                "or pass an image directly to the method."
            )
        if self._cached_masks is None:
            self._cached_masks = self._generate_masks(self._cached_image_rgb)
        return self._cached_image_rgb, self._cached_masks

    # ------------------------------------------------------------------
    # PUBLIC: Output methods
    # ------------------------------------------------------------------

    def get_coordinates(self, input=None):
        """
        Return bounding-box coordinates for each detected segment.

        Returns list of dicts with keys:
            segment_id, bbox, area, predicted_iou, stability_score
        """
        _, masks = self._get_cached_masks(input)
        return [
            {
                "segment_id":      i,
                "bbox":            mask["bbox"],
                "area":            mask["area"],
                "predicted_iou":   round(mask["predicted_iou"],   4),
                "stability_score": round(mask["stability_score"], 4),
            }
            for i, mask in enumerate(masks)
        ]

    def get_masks(self, input=None):
        """
        Return binary boolean masks for each detected segment.

        Returns list of dicts with keys:
            segment_id, mask (np.ndarray bool H×W), area
        """
        _, masks = self._get_cached_masks(input)
        return [
            {
                "segment_id": i,
                "mask":       mask["segmentation"].astype(bool),
                "area":       mask["area"],
            }
            for i, mask in enumerate(masks)
        ]

    def get_segments(self, input=None):
        """
        Return full-size RGBA PIL images for each detected segment.
        Background outside the mask is transparent.

        Returns list of dicts with keys:
            segment_id, image (PIL RGBA full size), bbox, area
        """
        image_rgb, masks = self._get_cached_masks(input)
        results = []
        for i, mask in enumerate(masks):
            mask_bool = mask["segmentation"].astype(bool)

            rgba = np.zeros((*image_rgb.shape[:2], 4), dtype=np.uint8)
            rgba[..., :3] = image_rgb
            rgba[..., 3]  = (mask_bool * 255).astype(np.uint8)

            results.append({
                "segment_id": i,
                "image":      Image.fromarray(rgba, mode="RGBA"),  # full size, no crop
                "bbox":       mask["bbox"],
                "area":       mask["area"],
            })
        return results

    def clear_cache(self):
        """Release cached image and masks, and free GPU memory."""
        self._cached_image_rgb = None
        self._cached_masks     = None
        torch.cuda.empty_cache()

print("SegmentationModule class defined.")

## Instantiate the Module

In [ ]:
seg = SegmentationModule()
seg.min_area_ratio       = 0.05  # keep segments >= 5% of largest
seg.min_mask_region_area = 500   # let smaller food regions through
seg.dedup_iou_threshold  = 0.5   # drop overlapping masks aggressively
print("Model loaded and ready.")

## Batch Processing Loop

For every image in the input folder the pipeline:
1. Saves **coordinates** as JSON files
2. Saves **masks** as `.txt` files (0/1 grid)
3. Saves **segments** as full-size transparent-background PNG files

In [ ]:
image_files = [
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
]

print(f"Found {len(image_files)} images in {INPUT_DIR}\n")

total_start = time.time()

for filename in image_files:
    image_path = os.path.join(INPUT_DIR, filename)
    base_name  = os.path.splitext(filename)[0]

    print(f"Processing: {filename}")
    t0 = time.time()

    # Per-image sub-folders
    img_coords_dir   = os.path.join(COORDS_DIR,   base_name)
    img_masks_dir    = os.path.join(MASKS_DIR,    base_name)
    img_segments_dir = os.path.join(SEGMENTS_DIR, base_name)
    for d in [img_coords_dir, img_masks_dir, img_segments_dir]:
        os.makedirs(d, exist_ok=True)

    # Load image once; results are cached for all three calls
    seg.set_image_from_path(image_path)

    # 1. COORDINATES → JSON
    coordinates = seg.get_coordinates()
    for coord in coordinates:
        sid      = coord["segment_id"]
        out_path = os.path.join(img_coords_dir, f"{base_name}_coord_{sid:03d}.json")
        with open(out_path, "w") as f:
            json.dump(coord, f, indent=4)
    print(f"  ✔ Coordinates saved  ({len(coordinates)} segments)")

    # 2. MASKS → TXT
    masks = seg.get_masks()
    for mask_data in masks:
        sid      = mask_data["segment_id"]
        out_path = os.path.join(img_masks_dir, f"{base_name}_mask_{sid:03d}.txt")
        np.savetxt(out_path, mask_data["mask"].astype(int), fmt="%d")
    print(f"  ✔ Masks saved        ({len(masks)} segments)")

    # 3. SEGMENTS → PNG
    segments = seg.get_segments()
    for segment in segments:
        sid      = segment["segment_id"]
        out_path = os.path.join(img_segments_dir, f"{base_name}_segment_{sid:03d}.png")
        segment["image"].save(out_path)
    print(f"  ✔ Segments saved     ({len(segments)} segments)")

    elapsed = time.time() - t0
    print(f"  ⏱  {elapsed:.2f}s\n")

    # Free GPU memory before next image
    seg.clear_cache()

total_elapsed = time.time() - total_start
print(f"All done! Total time: {total_elapsed:.2f}s")
print(f"Output written to: {OUTPUT_DIR}")

## Visualization: Input + Outputs Side-by-Side

In [ ]:
# ============================================================
# VISUALIZATION: Display Input Images alongside Their Segments
# ============================================================
import os
import json
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np

def display_input_with_segments(input_dir, segments_dir, coords_dir, supported_ext=('.jpg', '.jpeg', '.png')):
    """
    For each input image, display it side-by-side with all its segmented outputs.
    Bounding boxes from the coordinate JSONs are drawn on the original image.
    """
    image_files = sorted([
        f for f in os.listdir(input_dir)
        if f.lower().endswith(supported_ext)
    ])

    if not image_files:
        print("No input images found in", input_dir)
        return

    for filename in image_files:
        base_name   = os.path.splitext(filename)[0]
        image_path  = os.path.join(input_dir, filename)
        seg_folder  = os.path.join(segments_dir, base_name)
        coord_folder = os.path.join(coords_dir,  base_name)

        # Load original image
        original = Image.open(image_path).convert("RGB")

        # Load all segment PNGs (sorted by segment id)
        seg_files = sorted([
            f for f in os.listdir(seg_folder)
            if f.endswith(".png")
        ]) if os.path.isdir(seg_folder) else []

        # Load all coordinate JSONs
        coord_map = {}
        if os.path.isdir(coord_folder):
            for cf in os.listdir(coord_folder):
                if cf.endswith(".json"):
                    with open(os.path.join(coord_folder, cf)) as fp:
                        data = json.load(fp)
                    coord_map[data["segment_id"]] = data

        n_segs   = len(seg_files)
        n_cols   = 4                          # segments per row
        n_seg_rows = math.ceil(n_segs / n_cols) if n_segs else 1
        
        # ── Figure layout ──────────────────────────────────────────
        # Row 0  : original image (spans full width)
        # Row 1+ : segment thumbnails
        fig_height = 5 + n_seg_rows * 3.5
        fig, axes  = plt.subplots(
            1 + n_seg_rows, n_cols,
            figsize=(n_cols * 4, fig_height),
            gridspec_kw={"height_ratios": [5] + [3.5] * n_seg_rows}
        )

        # Make axes always 2-D
        if axes.ndim == 1:
            axes = axes["np.newaxis, :"]

        # ── Row 0: Original image with bounding boxes ───────────────
        # Merge top row into one axes using GridSpec tricks
        for ax in axes[0, 1:]:
            ax.set_visible(False)
        ax_orig = axes[0, 0]
        # Rebuild with a proper wide axes by redrawing on a GridSpec
        plt.close(fig)

        from matplotlib.gridspec import GridSpec
        total_rows = 1 + n_seg_rows
        fig = plt.figure(figsize=(n_cols * 4, fig_height))
        gs  = GridSpec(total_rows, n_cols, figure=fig,
                       height_ratios=[5] + [3.5] * n_seg_rows,
                       hspace=0.35, wspace=0.25)

        ax_orig = fig.add_subplot(gs[0, :])   # full-width top row
        ax_orig.imshow(original)
        ax_orig.set_title(f"INPUT: {filename}  ({original.width}×{original.height} px)",
                          fontsize=13, fontweight="bold", pad=8)
        ax_orig.axis("off")

        # Draw bounding boxes on the original
        colors = plt.cm.tab20.colors
        for sid, coord in coord_map.items():
            x, y, w, h = coord["bbox"]
            color = colors[sid % len(colors)]
            rect  = mpatches.Rectangle(
                (x, y), w, h,
                linewidth=2, edgecolor=color, facecolor="none"
            )
            ax_orig.add_patch(rect)
            ax_orig.text(x + 3, y - 6,
                         f"#{sid}  IoU:{coord['predicted_iou']:.2f}",
                         color="white", fontsize=7,
                         bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor="none"))

        # ── Rows 1+: Segments ───────────────────────────────────────
        seg_axes = [fig.add_subplot(gs[1 + r, c])
                    for r in range(n_seg_rows) for c in range(n_cols)]

        for idx, seg_file in enumerate(seg_files):
            # Parse segment id from filename  (e.g. burger_segment_003.png → 3)
            sid_str  = seg_file.replace(".png", "").split("_")[-1]
            sid      = int(sid_str) if sid_str.isdigit() else idx
            seg_path = os.path.join(seg_folder, seg_file)

            seg_img  = Image.open(seg_path).convert("RGBA")
            # Crop to bounding box to remove excess whitespace
            if sid in coord_map:
                x, y, w, h = [int(v) for v in coord_map[sid]["bbox"]]
                x2, y2 = min(x + w, seg_img.width), min(y + h, seg_img.height)
                seg_img = seg_img.crop((x, y, x2, y2))

            ax = seg_axes[idx]
            ax.imshow(seg_img)
            area = coord_map[sid]["area"] if sid in coord_map else "?"
            ax.set_title(f"Seg #{sid}  area:{area}", fontsize=8)
            ax.axis("off")

        # Hide unused segment axes
        for idx in range(n_segs, len(seg_axes)):
            seg_axes[idx].set_visible(False)

        plt.suptitle(f"{base_name} — {n_segs} segments detected",
                     fontsize=14, fontweight="bold", y=1.01)
        plt.tight_layout()
        plt.show()
        print("-" * 70)

# ── Run it ─────────────────────────────────────────────────────────────
display_input_with_segments(
    input_dir    = INPUT_DIR,
    segments_dir = SEGMENTS_DIR,
    coords_dir   = COORDS_DIR,
)


In [ ]:
# Sanity check: list all output files
!find /kaggle/working/capstone-demonstration-outputs -type f | sort

## Output Structure
```
capstone-demonstration-outputs/
├── coordinates/
│   └── <image_name>/
│       ├── <image_name>_coord_000.json
│       └── ...
├── masks/
│   └── <image_name>/
│       ├── <image_name>_mask_000.txt
│       └── ...
└── segments/
    └── <image_name>/
        ├── <image_name>_segment_000.png
        └── ...
```